# Task 4 — style-conditioned photo-to-sketch cGAN
Choose **Runtime → Change runtime type → GPU**, then **Run all**. Allow Drive access and upload **task4_source.zip** when prompted. No previous trained-model upload is required; Task 4 trains from scratch.

Dataset: the prepared 2,104 Flowers photos paired with three deterministic synthetic targets: fine pencil, technical ink and tonal charcoal. All styles of a photo stay in the same split. Training: 1,486 photos / 4,458 pairs; validation: 262 photos / 786 pairs; test: 356 photos / 1,068 pairs. The test set is reserved.

Generator: a five-level U-Net with learned categorical style embedding and paired skips. Discriminator: a 70-pixel PatchGAN conditioned on photo, sketch and learned style embedding. Loss: adversarial BCE plus lambda_L1 × paired pixel L1. Spatial flips are applied identically to photo and target.

Optuna starting budget: **6 trials × 5 epochs**, then **60 final epochs** for the chosen configuration. It tunes both learning rates, batch size, base channels, dropout, embedding dimension and lambda_L1. Separate GAN loss components, fixed validation sample panels and metrics are recorded in MLflow.

Backups: `MyDrive/GenAI_Ass1/task4_v1`. Reconnect and rerun with the same configuration/budget to restore the study and resume the last completed final epoch. The notebook evaluates validation only, exports the generator to ONNX with a selectable style input, and downloads **task4_results.zip**.

The synthetic targets are algorithmic approximations, not human-drawn sketches. Inspect generated detail, style diversity and the blank-white baseline; numeric scores alone can hide nearly empty outputs.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, zipfile, shutil, subprocess, sys, json, sqlite3
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose a GPU runtime, then reconnect before training.')
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
PROJECT = Path('/content/ass1_task4')
BACKUP = Path('/content/drive/MyDrive/GenAI_Ass1/task4_v1')
PROJECT.mkdir(exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)


In [ ]:
uploaded = files.upload()  # Select colab/task4_source.zip
if len(uploaded) != 1:
    raise ValueError('Upload exactly the generated source ZIP.')
archive_name = next(iter(uploaded))
with zipfile.ZipFile(archive_name) as archive:
    for entry in archive.infolist():
        destination = (PROJECT / entry.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Invalid ZIP path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
# Keep Colab's CUDA-enabled PyTorch installation; do not install the CPU wheel.
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)

config = json.loads(Path('configs/task4.json').read_text())
print('Task 4 configuration:', json.dumps(config, indent=2))


In [ ]:
# Restore previous results after reconnecting, and define consistent backup helpers.
for folder in ['runs', 'models', 'mlruns']:
    if (BACKUP / folder).exists():
        shutil.copytree(BACKUP / folder, PROJECT / folder, dirs_exist_ok=True)
if (BACKUP / 'mlflow.db').exists():
    shutil.copy2(BACKUP / 'mlflow.db', PROJECT / 'mlflow.db')

def backup_results():
    ignore = shutil.ignore_patterns('*.sqlite3', '*.db', '*.tmp', '*-journal', '*-wal', '*-shm')
    for folder in ['runs', 'models', 'mlruns']:
        if (PROJECT / folder).exists():
            shutil.copytree(PROJECT / folder, BACKUP / folder, dirs_exist_ok=True, ignore=ignore)
    databases = list((PROJECT / 'runs').rglob('*.sqlite3')) if (PROJECT / 'runs').exists() else []
    if (PROJECT / 'mlflow.db').exists():
        databases.append(PROJECT / 'mlflow.db')
    for source in databases:
        local_copy = Path('/content') / (source.name + '.backup')
        local_copy.unlink(missing_ok=True)
        with sqlite3.connect(str(source), timeout=60) as src, sqlite3.connect(str(local_copy)) as dest:
            src.backup(dest)
        destination = BACKUP / source.relative_to(PROJECT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(local_copy, destination)

def run(arguments):
    process = subprocess.Popen([sys.executable, '-u', *arguments], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, cwd=PROJECT)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(record, dict) and 'epoch' in record:
                backup_results()
        if process.wait() != 0:
            raise RuntimeError('Command failed; inspect the output above.')
    except BaseException:
        process.terminate()
        process.wait()
        backup_results()
        raise
    backup_results()


In [ ]:
# Download data to local runtime storage; preserve official split definitions.
import urllib.request, tarfile
raw = PROJECT / 'raw_data'
raw.mkdir(exist_ok=True)
for filename in ['102flowers.tgz', 'setid.mat']:
    path = raw / filename
    if not path.exists():
        temporary = raw / (filename + '.part')
        urllib.request.urlretrieve('https://www.robots.ox.ac.uk/~vgg/data/flowers/102/' + filename, temporary)
        temporary.replace(path)
if not (raw / 'jpg' / 'image_08189.jpg').exists():
    with tarfile.open(raw / '102flowers.tgz') as archive:
        archive.extractall(raw, filter='data')
run(['prepare_dataset.py'])
# Verify photo grouping and style pairing match the local prepared foundation.
import hashlib
manifest = Path('data/prepared_v2/sketch/paired_samples.json')
expected_data = json.loads(Path('experiments/task4/data.json').read_text())
assert hashlib.sha256(manifest.read_bytes()).hexdigest() == expected_data['manifest_sha256']
print('Paired sketch manifest matches:', expected_data)
run(['-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## Tune the conditional GAN


In [ ]:
import optuna
SEARCH = Path('runs/task4/search')
FINAL = Path('runs/task4/final')
TRIAL_BUDGET = 6
TRIAL_EPOCHS = 5
TOTAL_EPOCHS = 60
# These are editable starting budgets. Keep them unchanged when resuming.
finished = 0
study_path = SEARCH / 'study.sqlite3'
if study_path.exists():
    study = optuna.load_study(study_name='task4', storage='sqlite:///' + str(study_path.resolve()))
    finished = sum(t.state in [optuna.trial.TrialState.COMPLETE,optuna.trial.TrialState.PRUNED] for t in study.trials)
remaining = max(0,TRIAL_BUDGET-finished)
if remaining:
    run(['-m','task4.train','tune','--config','configs/task4.json','--epochs',str(TRIAL_EPOCHS),
         '--trials',str(remaining),'--device','cuda','--workers','2','--output',str(SEARCH)])
print(json.loads((SEARCH / 'best_config.json').read_text()))


## Train the selected GAN from scratch, or resume this final run


In [ ]:
selected = SEARCH / 'best_config.json'
chosen = json.loads(selected.read_text())
chosen['epochs'] = TOTAL_EPOCHS
last = FINAL / 'last.pt'
completed = 0
if last.exists():
    saved = torch.load(last,map_location='cpu',weights_only=True)
    assert saved['config'] == chosen, 'Selected configuration changed; use a new experiment folder.'
    completed = saved['epoch']
if completed < TOTAL_EPOCHS:
    arguments = ['-m','task4.train','train','--config',str(selected),'--epochs',str(TOTAL_EPOCHS),
                 '--device','cuda','--workers','2','--output',str(FINAL)]
    if last.exists():
        arguments += ['--resume',str(last)]
    run(arguments)
else:
    assert (FINAL / 'best.pt').exists()
    print('Final training completed:',completed,'epochs')


## Validate all three styles


In [ ]:
run(['-m','task4.evaluate','--checkpoint',str(FINAL / 'best.pt'),'--split','val',
     '--device','cuda','--output','runs/task4/validation'])
from IPython.display import display,Image
import pandas as pd
scores = pd.read_csv('runs/task4/validation/grouped_metrics.csv')
display(scores.round(4))
report = json.loads(Path('runs/task4/validation/results.json').read_text())
print('Generated style diversity vs target style diversity:',report['style_diversity'])
for panel in sorted(Path('runs/task4/validation').glob('example_*.png'))[:12]:
    display(Image(filename=str(panel)))
for panel in sorted(Path('runs/task4/validation').glob('failure_candidate_*.png')):
    display(Image(filename=str(panel)))
# Compare fixed validation photos throughout training for style/detail development.
for panel in sorted(FINAL.glob('samples_epoch_*.png'))[::max(1,len(list(FINAL.glob('samples_epoch_*.png')))//4)]:
    display(Image(filename=str(panel)))


## Export a single generator with image and style inputs


In [ ]:
run(['-m','task4.export','--checkpoint',str(FINAL / 'best.pt'),
     '--output','models/task4/sketch_generator.onnx'])
from backend.task4_runtime import Task4Runtime
from task4.model import load_checkpoint
from dataset_loaders import SketchDataset
import numpy as np
runtime = Task4Runtime('models/task4/sketch_generator.onnx')
model,_ = load_checkpoint(FINAL / 'best.pt')
model.eval()
counts = [0,0,0]
with torch.no_grad():
    for item in SketchDataset(split='val',augment=False):
        style = item['style']
        if counts[style] >= 4:
            continue
        x = item['input'][None]
        expected = model(x,torch.tensor([style],dtype=torch.long)).numpy()
        actual = runtime.run(x.numpy(),style)
        np.testing.assert_allclose(actual,expected,rtol=1e-4,atol=2e-5)
        counts[style] += 1
        if counts == [4,4,4]:
            break
assert counts == [4,4,4]
print('ONNX style-input parity passed on four validation photos for each style.')
backup_results()


## Download models and experiment evidence


In [ ]:
backup_results()
output = Path('/content/task4_results.zip')
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as archive:
    for folder in ['models','runs','mlruns','experiments/task4']:
        for path in (PROJECT / folder).rglob('*'):
            if path.is_file() and not path.name.endswith(('-journal','-wal','-shm','.tmp','.sqlite3')):
                archive.write(path,path.relative_to(PROJECT))
            elif path.is_file() and path.suffix == '.sqlite3':
                archive.write(BACKUP / path.relative_to(PROJECT),path.relative_to(PROJECT))
    archive.write(PROJECT / 'configs/task4.json','configs/task4.json')
    if (BACKUP / 'mlflow.db').exists():
        archive.write(BACKUP / 'mlflow.db','mlflow.db')
files.download(str(output))
